# Практикум по теме «Дескрипторы: протокол дескрипторов, __get__, __set__, property под капотом»

## 🎯 Цель практикума
Закрепить понимание механизма дескрипторов в Python, научиться создавать свои дескрипторы для валидации, ленивой инициализации и других задач, а также разобраться в том, как устроен `@property`.

## 📋 Структура
1. **Базовые задачи** (3 шт.) – отработка основ протокола дескрипторов
2. **Задачи повышенной сложности** (2 шт.) – применение дескрипторов в реальных сценариях
3. **Домашнее задание** (3 шт.) – для самостоятельной работы

> 💡 **Совет:** перед выполнением задач повторите порядок поиска атрибутов и различия между data- и non-data дескрипторами.

## 🟢 Базовые задачи

### Задача 1. Простейший дескриптор-логгер
**Условие:** Реализуйте дескриптор `Logger`, который при каждом доступе (чтении или записи) выводит сообщение в консоль. Сообщение должно содержать имя атрибута и действие (чтение/запись). Для хранения значения используйте `obj.__dict__`.

**Требования:**
- Дескриптор должен быть **data-дескриптором** (реализовать `__get__` и `__set__`).
- Имя атрибута передаётся в конструктор дескриптора.
- При чтении выводится: `[LOG] Чтение <имя> = <значение>`
- При записи: `[LOG] Запись <имя> = <значение>`

```python
# Пример использования:
class User:
    name = Logger("name")
    age = Logger("age")

    def __init__(self, name, age):
        self.name = name
        self.age = age

u = User("Alice", 30)
print(u.name)   # [LOG] Чтение name = Alice
u.age = 31      # [LOG] Запись age = 31

In [ ]:
# Шаблон для решения
class Logger:
    def __init__(self, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        # Реализуйте чтение с логированием
        pass

    def __set__(self, obj, value):
        # Реализуйте запись с логированием
        pass

# Проверка
class Test:
    attr = Logger("attr")
t = Test()
t.attr = 42
print(t.attr)

### Задача 2. Дескриптор с валидацией типа
**Условие:** Создайте дескриптор `Typed`, который проверяет, что присваиваемое значение имеет заданный тип. Если тип не совпадает, выбрасывается `TypeError`. Имя атрибута и ожидаемый тип передаются в конструктор.

**Требования:**
- Используйте `isinstance()` для проверки.
- Храните значение в `obj.__dict__` под именем дескриптора.

```python
# Пример использования:
class Person:
    name = Typed(str, "name")
    age = Typed(int, "age")

p = Person()
p.name = "Ivan"   # OK
p.age = 25        # OK
# p.age = "25"    # TypeError: age должен быть int

In [ ]:
# Шаблон для решения
class Typed:
    def __init__(self, expected_type, name):
        self.expected_type = expected_type
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        # Проверьте тип и установите значение
        pass

# Проверка
class Test:
    value = Typed(int, "value")
t = Test()
t.value = 100
print(t.value)
# t.value = "string"  # TypeError

### Задача 3. Дескриптор для положительного числа
**Условие:** Реализуйте дескриптор `Positive`, который гарантирует, что значение атрибута является положительным числом (целым или вещественным). Если значение не положительное, выбрасывается `ValueError`. Если значение не является числом, выбрасывается `TypeError`.

**Требования:**
- Поддерживает как `int`, так и `float`.
- Для проверки на число используйте `isinstance(value, (int, float))`.

```python
# Пример использования:
class Account:
    balance = Positive("balance")

acc = Account()
acc.balance = 1000.50   # OK
# acc.balance = -50     # ValueError
# acc.balance = "money" # TypeError

In [ ]:
# Шаблон для решения
class Positive:
    def __init__(self, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        # Реализуйте проверки
        pass

# Проверка
class Test:
    num = Positive("num")
t = Test()
t.num = 5.5
print(t.num)
# t.num = -1  # ValueError

## 🔴 Задачи повышенной сложности

### Задача 4. Дескриптор с ленивой загрузкой (LazyProperty)
**Условие:** Напишите дескриптор `LazyProperty`, который вычисляет значение функции при первом обращении и кэширует его. При повторных обращениях возвращается закэшированное значение, без повторного вызова функции. Дескриптор должен работать аналогично `@property`, но с ленивым вычислением.

**Подсказка:** В `__get__` вызовите функцию, сохраните результат в `obj.__dict__` под тем же именем, что и дескриптор, а затем верните его.

```python
# Пример использования:
class Data:
    def __init__(self, data):
        self.data = data

    @LazyProperty
    def processed(self):
        print("Вычисляем processed...")
        return [x * 2 for x in self.data]

d = Data([1, 2, 3])
print(d.processed)   # Вычисляем processed... [2, 4, 6]
print(d.processed)   # [2, 4, 6] (без повторного вычисления)

In [ ]:
# Шаблон для решения
class LazyProperty:
    def __init__(self, func):
        self.func = func
        self.name = func.__name__

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        # Реализуйте ленивое вычисление и кэширование
        pass

# Проверка
class Test:
    @LazyProperty
    def expensive(self):
        print("Вычисление...")
        return 42

t = Test()
print(t.expensive)  # Вычисление... 42
print(t.expensive)  # 42

### Задача 5. Дескриптор с ограничением на количество записей
**Условие:** Реализуйте дескриптор `WriteOnce`, который позволяет установить значение атрибута только один раз. При повторной попытке записи должно выбрасываться исключение `PermissionError` (или просто игнорироваться – по вашему выбору). Дескриптор должен быть **data-дескриптором**.

**Требования:**
- Храните флаг, было ли значение уже установлено.
- Значение хранится в `obj.__dict__`, но запись разрешена только если его там ещё нет.

```python
# Пример использования:
class Config:
    api_key = WriteOnce("api_key")

cfg = Config()
cfg.api_key = "secret"
print(cfg.api_key)        # secret
# cfg.api_key = "new"     # PermissionError: Cannot change api_key

In [ ]:
class WriteOnce:
    def __init__(self, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        # Проверьте, существует ли уже значение
        pass

# Проверка
class Test:
    attr = WriteOnce("attr")
t = Test()
t.attr = "first"
print(t.attr)
# t.attr = "second"  # должно вызвать ошибку

## 🏠 Домашнее задание

### Задача 6. Дескриптор для строки фиксированной длины
**Условие:** Создайте дескриптор `FixedString`, который принимает максимальную длину строки. При присваивании строки, если её длина превышает максимальную, обрезайте её до заданной длины (или выбрасывайте исключение – на ваш выбор). Также убедитесь, что значение является строкой.

```python
# Пример использования:
class User:
    username = FixedString(max_length=10, name="username")

u = User()
u.username = "very_long_username"
print(u.username)   # "very_long_"

### Задача 7. Дескриптор с валидацией по регулярному выражению
**Условие:** Реализуйте дескриптор `Regex`, который проверяет, что присваиваемая строка соответствует заданному регулярному выражению (передаётся в конструктор). Если не соответствует, выбрасывается `ValueError`. Используйте модуль `re`.

```python
# Пример использования:
import re
class Contact:
    phone = Regex(r'^\+7\d{10}$', "phone")
    email = Regex(r'^[\w\.-]+@[\w\.-]+\.\w+$', "email")

c = Contact()
c.phone = "+71234567890"   # OK
# c.phone = "123"          # ValueError

### Задача 8. Дескриптор с предупреждением об устаревании (Deprecated)
**Условие:** Создайте дескриптор `Deprecated`, который при чтении или записи выводит предупреждение (используйте модуль `warnings`), что атрибут устарел и будет удалён в будущем. При этом дескриптор должен вести себя как обычный (хранить значение). Используйте `warnings.warn()`.

```python
# Пример использования:
class OldAPI:
    old_method = Deprecated("old_method")

api = OldAPI()
api.old_method = "some value"  # DeprecationWarning: old_method is deprecated
print(api.old_method)          # DeprecationWarning: old_method is deprecated

## 📌 Полезные подсказки

- В дескрипторах всегда помните о `if obj is None:` в `__get__` – это нужно для доступа через класс.
- Для хранения значений используйте `obj.__dict__`, чтобы не создавать лишние атрибуты.
- Data-дескрипторы имеют приоритет над атрибутами экземпляра, поэтому если вы хотите разрешить переопределение, используйте non-data (только `__get__`).
- Для ленивых свойств полезно после вычисления заменять дескриптор на обычный атрибут: `obj.__dict__[self.name] = value`.

Удачи в решении! 🚀